# Pretrained Word-Vector Sentence-Pair Experiments

Research notebook for issue #57 using only pretrained multilingual fastText word vectors. It evaluates tokenization, lemmatization, stop-word handling, sentence-vector aggregation, and multiple classifier families. Runs are logged to MLflow immediately, including the fitted classifier artifact and metadata, so the notebook does not keep trained model packages in memory.

In [ ]:
import gc
import json
import os
import re
import urllib.request
from itertools import product
from pathlib import Path
from tempfile import TemporaryDirectory

import joblib
import mlflow
import numpy as np
import pandas as pd
import simplemma
from gensim.models.fasttext import load_facebook_vectors
from gensim.models.keyedvectors import KeyedVectors
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import ParameterGrid, train_test_split
from sklearn.naive_bayes import GaussianNB
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import LinearSVC
from sklearn.tree import DecisionTreeClassifier

os.environ.setdefault('MPLCONFIGDIR', '/tmp/matplotlib')

try:
    from xgboost import XGBClassifier
except ImportError:
    XGBClassifier = None

try:
    from lightgbm import LGBMClassifier
except ImportError:
    LGBMClassifier = None

from helpers.data import load_processed_data, load_sample_submission
from helpers.metrics import compute_all_metrics
from helpers.mlflow import log_common_context, log_metrics, log_model_artifacts, start_notebook_run
from helpers.submission import build_submission, save_submission

## Load Data

Prefer DVC-versioned processed splits. If this notebook is copied into Kaggle, fallback paths can be used.

In [ ]:
PROCESSED_DIR = Path('../data/processed')
RAW_DIR = Path('../data/raw')
KAGGLE_INPUT_DIR = Path('/kaggle/input/competitions/contradictory-my-dear-watson')

if (PROCESSED_DIR / 'train_split.csv').exists():
    train_df, val_df, test_df = load_processed_data(PROCESSED_DIR)
    sample_submission = load_sample_submission(RAW_DIR / 'sample_submission.csv')
    data_source = 'dvc_processed_split'
elif (KAGGLE_INPUT_DIR / 'train.csv').exists():
    full_train_df = pd.read_csv(KAGGLE_INPUT_DIR / 'train.csv')
    test_df = pd.read_csv(KAGGLE_INPUT_DIR / 'test.csv')
    sample_submission = pd.read_csv(KAGGLE_INPUT_DIR / 'sample_submission.csv')
    train_df, val_df = train_test_split(
        full_train_df,
        test_size=0.2,
        random_state=42,
        stratify=full_train_df['label'],
    )
    data_source = 'kaggle_input_fallback_split'
else:
    raise FileNotFoundError('Could not find DVC processed splits or Kaggle input files.')

train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

dataset_languages = (
    pd.concat([train_df[['language', 'lang_abv']], val_df[['language', 'lang_abv']], test_df[['language', 'lang_abv']]])
    .drop_duplicates()
    .sort_values('lang_abv')
    .reset_index(drop=True)
)

len(train_df), len(val_df), len(test_df), data_source, dataset_languages

## Experiment Controls

The dataset languages are covered by fastText Common Crawl + Wikipedia vectors for 157 languages: https://fasttext.cc/docs/en/crawl-vectors.html. The original `vec_lookup` mode uses `cc.{lang}.300.vec.gz` / `.vec` files. A separate `bin_subword` mode uses `cc.{lang}.300.bin.gz` / `.bin` files so fastText can infer vectors for unseen words from subword n-grams. Constants below define the whole search. If you change them, the notebook logs the effective config to MLflow, so the run is still self-describing without a commit just for constant changes.

In [ ]:
RANDOM_STATE = 42
MLFLOW_LOGGING = True
STAGE_1_MAX_FEATURE_EXPERIMENTS = None
STAGE_2_MAX_MODEL_EXPERIMENTS = None

# NEW: skip first N experiments (set to 0 to run all)
SKIP_FIRST_N_EXPERIMENTS = 0

# NEW: compress downloaded bin_subword fastText models to save HDD/RAM footprint.
# Requires `pip install compress-fasttext` (optional; graceful fallback provided).
ENABLE_COMPRESS_FASTTEXT = True
PRETRAINED_FASTTEXT_COMPRESSED_SUFFIX = '.compressed.bin'

# NEW: disable fastText subword inference for OOV tokens in bin_subword mode
SKIP_SUBWORD_OOV = True

# NEW: disable fastText subword-based experiments entirely
ENABLE_BIN_SUBWORD_EXPERIMENTS = False

BASE_FEATURE_MODEL_FAMILY = 'linear_svm'
BASE_FEATURE_MODEL_PARAMS = {'C': 1.0}

PRETRAINED_FASTTEXT_DIR = Path('../embeddings/fasttext')
DOWNLOAD_PRETRAINED_FASTTEXT = True
PRETRAINED_FASTTEXT_MAX_VECTORS = 200_000
PRETRAINED_FASTTEXT_LANGS = sorted(pd.concat([train_df, val_df, test_df])['lang_abv'].dropna().unique().tolist())
PRETRAINED_FASTTEXT_BASE_URL = 'https://dl.fbaipublicfiles.com/fasttext/vectors-crawl'
EMBEDDING_MODES = ['vec_lookup', 'bin_subword']
PRETRAINED_FASTTEXT_VEC_URLS = {
    lang: f'{PRETRAINED_FASTTEXT_BASE_URL}/cc.{lang}.300.vec.gz'
    for lang in PRETRAINED_FASTTEXT_LANGS
}
PRETRAINED_FASTTEXT_BIN_URLS = {
    lang: f'{PRETRAINED_FASTTEXT_BASE_URL}/cc.{lang}.300.bin.gz'
    for lang in PRETRAINED_FASTTEXT_LANGS
}

PREPROCESS_OPTIONS = {
    'token_strategy': ['raw', 'lemma'],
    'stop_strategy': ['keep_all', 'remove_stopwords', 'remove_stopwords_keep_negation'],
}

AGGREGATION_METHODS = [
    'mean_pair',
    'tfidf_mean_pair',
    'diff_product',
    'cosine_only',
    'all_features',
]

OOV_FEATURE_MODES = [
    'none',
    'coverage_stats',
]

MODEL_PARAM_GRIDS = {
    'logistic_regression': {'C': [0.5, 1.0, 2.0]},
    'linear_svm': {'C': [0.5, 1.0, 2.0]},
    'random_forest': {'n_estimators': [300], 'max_depth': [None, 20], 'min_samples_leaf': [1, 3]},
    'decision_tree': {'max_depth': [20, 50, None], 'min_samples_leaf': [1, 3]},
    'gaussian_nb': {'var_smoothing': [1e-9, 1e-8, 1e-7]},
    'xgboost': {'n_estimators': [300], 'max_depth': [3, 5], 'learning_rate': [0.05, 0.1]},
    'lightgbm': {'n_estimators': [300], 'num_leaves': [31, 63], 'learning_rate': [0.05, 0.1]},
}

EXPERIMENT_CONFIG = {
    'random_state': RANDOM_STATE,
    'stage_1_max_feature_experiments': STAGE_1_MAX_FEATURE_EXPERIMENTS,
    'stage_2_max_model_experiments': STAGE_2_MAX_MODEL_EXPERIMENTS,
    'base_feature_model_family': BASE_FEATURE_MODEL_FAMILY,
    'base_feature_model_params': BASE_FEATURE_MODEL_PARAMS,
    'preprocess_options': PREPROCESS_OPTIONS,
    'embedding_modes': EMBEDDING_MODES,
    'aggregation_methods': AGGREGATION_METHODS,
    'oov_feature_modes': OOV_FEATURE_MODES,
    'model_param_grids': MODEL_PARAM_GRIDS,
    'pretrained_fasttext_max_vectors': PRETRAINED_FASTTEXT_MAX_VECTORS,
    'pretrained_fasttext_vec_urls': PRETRAINED_FASTTEXT_VEC_URLS,
    'pretrained_fasttext_bin_urls': PRETRAINED_FASTTEXT_BIN_URLS,
}

## Tokenization and Preprocessing

Tokenization is Unicode-aware so non-Latin dataset languages are not discarded. Lemmatization uses `simplemma` where that library supports the dataset language; unsupported languages are left unchanged and recorded in MLflow metadata.

In [ ]:
NEGATION_WORDS = {'no', 'not', 'nor', 'never', 'none', 'nobody', 'nothing', 'neither', 'cannot', "can't", "won't", "n't"}
TOKEN_PATTERN = re.compile(r"\w+(?:'\w+)?", flags=re.UNICODE)
SIMPLEMMA_SUPPORTED_LANGS = {
    'bg', 'de', 'el', 'en', 'es', 'fr', 'hi', 'ru', 'sw', 'tr'
}
SIMPLEMMA_UNSUPPORTED_DATASET_LANGS = sorted(set(PRETRAINED_FASTTEXT_LANGS) - SIMPLEMMA_SUPPORTED_LANGS)


def normalize_text(value):
    return ' '.join(str(value).lower().split())


def tokenize(value):
    return TOKEN_PATTERN.findall(normalize_text(value))


def lemmatize_tokens(tokens, lang_abv):
    if lang_abv not in SIMPLEMMA_SUPPORTED_LANGS:
        return tokens
    return [simplemma.lemmatize(token, lang=lang_abv) for token in tokens]


def apply_preprocessing(tokens, lang_abv, token_strategy, stop_strategy):
    if token_strategy == 'lemma':
        tokens = lemmatize_tokens(tokens, lang_abv)

    if stop_strategy == 'remove_stopwords':
        tokens = [token for token in tokens if token not in ENGLISH_STOP_WORDS]
    elif stop_strategy == 'remove_stopwords_keep_negation':
        tokens = [token for token in tokens if token not in ENGLISH_STOP_WORDS or token in NEGATION_WORDS]
    elif stop_strategy != 'keep_all':
        raise ValueError(f'Unknown stop strategy: {stop_strategy}')

    return tokens


def tokenize_frame(frame, token_strategy, stop_strategy):
    premise_tokens = [
        apply_preprocessing(tokenize(text), lang_abv, token_strategy, stop_strategy)
        for text, lang_abv in zip(frame['premise'].fillna(''), frame['lang_abv'])
    ]
    hypothesis_tokens = [
        apply_preprocessing(tokenize(text), lang_abv, token_strategy, stop_strategy)
        for text, lang_abv in zip(frame['hypothesis'].fillna(''), frame['lang_abv'])
    ]
    return pd.DataFrame({
        'premise_tokens': premise_tokens,
        'hypothesis_tokens': hypothesis_tokens,
    })


def tokens_to_text(token_rows):
    return [' '.join(tokens) for tokens in token_rows]


def fit_tfidf_weights(tokenized_train):
    corpus = tokens_to_text(tokenized_train['premise_tokens']) + tokens_to_text(tokenized_train['hypothesis_tokens'])
    vectorizer = TfidfVectorizer(token_pattern=r'(?u)\b\w+\b')
    vectorizer.fit(corpus)
    weights = dict(zip(vectorizer.get_feature_names_out(), vectorizer.idf_))
    return vectorizer, weights

## Pretrained Vector Feature Builder

The builder loads one language vector file at a time, fills train/validation/test feature matrices for that language, then releases the keyed vectors. This avoids keeping all pretrained embedding models in RAM.

In [ ]:
def fasttext_vector_path(lang, embedding_mode):
    if embedding_mode == 'vec_lookup':
        candidates = [
            PRETRAINED_FASTTEXT_DIR / f'cc.{lang}.300.vec.gz',
            PRETRAINED_FASTTEXT_DIR / f'cc.{lang}.300.vec',
        ]
    elif embedding_mode == 'bin_subword':
        candidates = [
            PRETRAINED_FASTTEXT_DIR / f'cc.{lang}.300{PRETRAINED_FASTTEXT_COMPRESSED_SUFFIX}',
            PRETRAINED_FASTTEXT_DIR / f'cc.{lang}.300.bin.gz',
            PRETRAINED_FASTTEXT_DIR / f'cc.{lang}.300.bin',
        ]
    else:
        raise ValueError(f'Unknown embedding mode: {embedding_mode}')
    for path in candidates:
        if path.exists():
            return path
    return candidates[0]


def fasttext_download_url(lang, embedding_mode):
    if embedding_mode == 'vec_lookup':
        return PRETRAINED_FASTTEXT_VEC_URLS[lang]
    if embedding_mode == 'bin_subword':
        return PRETRAINED_FASTTEXT_BIN_URLS[lang]
    raise ValueError(f'Unknown embedding mode: {embedding_mode}')


def download_fasttext_vectors(lang, embedding_mode):
    PRETRAINED_FASTTEXT_DIR.mkdir(parents=True, exist_ok=True)
    destination = fasttext_vector_path(lang, embedding_mode)
    # If destination is the compressed candidate but not present, fall back to download original bin and later compress
    if destination.exists():
        return destination

    # If embedding_mode is compressed candidate but it isn't present, download original source
    if embedding_mode == 'bin_subword' and destination.name.endswith(PRETRAINED_FASTTEXT_COMPRESSED_SUFFIX):
        # choose raw bin.gz as initial download target
        destination = PRETRAINED_FASTTEXT_DIR / f'cc.{lang}.300.bin.gz'

    if destination.exists():
        return destination
    url = fasttext_download_url(lang, embedding_mode)
    print(f'Downloading {url} -> {destination}')
    urllib.request.urlretrieve(url, destination)

    # If bin_subword and compression enabled, try to compress to save disk space
    if embedding_mode == 'bin_subword' and ENABLE_COMPRESS_FASTTEXT:
        try:
            # perform compression locally and write compressed model next to downloaded file
            try:
                import compress_fasttext
            except Exception:
                compress_fasttext = None

            if compress_fasttext is None:
                print('compress-fasttext not installed; skipping compression step.')
                return destination

            # load original facebook model (may be large; we free it after compression)
            from gensim.models.fasttext import load_facebook_model
            print(f'Compressing {destination} to reduce disk usage (this may take time)...')
            fb_model = load_facebook_model(str(destination))
            big_kv = fb_model.wv
            # prune_ft_freq + product quantization recommended combination
            small_kv = compress_fasttext.prune_ft_freq(big_kv, pq=True)
            compressed_path = PRETRAINED_FASTTEXT_DIR / f'cc.{lang}.300{PRETRAINED_FASTTEXT_COMPRESSED_SUFFIX}'
            small_kv.save(str(compressed_path))
            # attempt to delete the original large file to free HDD
            try:
                os.remove(destination)
            except Exception as e:
                print(f'Warning: could not remove original file {destination}: {e}')
            # free memory
            try:
                del fb_model, big_kv, small_kv
            except Exception:
                pass
            gc.collect()
            print(f'Compressed model saved to {compressed_path}')
            return compressed_path
        except Exception as e:
            # Compression failed for some reason; keep the downloaded file and continue
            print(f'Warning: compress-fasttext compression failed for {destination}: {e}. Keeping original file.')
            return destination

    return destination


def is_fasttext_binary_path(path):
    suffixes = ''.join(Path(path).suffixes)
    return '.bin' in suffixes

from functools import lru_cache

@lru_cache(maxsize=128)
def load_fasttext_vectors(path, embedding_mode):
    # Handle compressed models produced by compress-fasttext
    if str(path).endswith(PRETRAINED_FASTTEXT_COMPRESSED_SUFFIX):
        try:
            import compress_fasttext
            # CompressedFastTextKeyedVectors provides a load method compatible with gensim keyedvectors
            return compress_fasttext.models.CompressedFastTextKeyedVectors.load(str(path)), 'bin_subword_compressed'
        except Exception as e:
            print(f'Warning: failed to load compressed fasttext model {path}: {e}. Falling back to normal loaders.')

    if embedding_mode == 'bin_subword':
        if not is_fasttext_binary_path(path):
            raise ValueError(f'bin_subword mode requires a .bin/.bin.gz/.compressed.bin file, got {path}')

        # Try loading raw .bin or .bin.gz directly first
        try:
            return load_facebook_vectors(str(path)), 'bin_subword'
        except AssertionError as e:
            # Only decompress on AssertionError as fallback
            print(f'Warning: AssertionError loading {path}: {e}. Trying decompression fallback...')

            import gzip, shutil, tempfile
            suffixes = ''.join(Path(path).suffixes)

            if not suffixes.endswith('.gz'):
                # Not gzipped and still failed; re-raise
                raise

            # Decompress .gz to temporary .bin and retry
            tmp = tempfile.NamedTemporaryFile(delete=False, suffix='.bin')
            tmp_name = tmp.name
            tmp.close()

            try:
                print(f'Decompressing to temporary {tmp_name}...')
                with gzip.open(path, 'rb') as fin, open(tmp_name, 'wb') as fout:
                    shutil.copyfileobj(fin, fout)
                kv = load_facebook_vectors(tmp_name)
                return kv, 'bin_subword'
            finally:
                try:
                    os.remove(tmp_name)
                except Exception:
                    pass

    if embedding_mode != 'vec_lookup':
        raise ValueError(f'Unknown embedding mode: {embedding_mode}')
    return KeyedVectors.load_word2vec_format(
        str(path),
        binary=False,
        limit=PRETRAINED_FASTTEXT_MAX_VECTORS,
    ), 'vec_lookup_only'


def validate_pretrained_fasttext_files(langs, embedding_modes):
    missing = []
    for embedding_mode in embedding_modes:
        for lang in langs:
            path = fasttext_vector_path(lang, embedding_mode)
            if not path.exists():
                missing.append((embedding_mode, lang, str(path), fasttext_download_url(lang, embedding_mode)))
    return missing


def lookup_token_vector(token, keyed_vectors, vector_source):
    if vector_source == 'bin_subword':
        # Skip subword-only OOV tokens instead of using inferred vectors.
        if SKIP_SUBWORD_OOV and token not in keyed_vectors.key_to_index:
            return None, False, False
        try:
            return keyed_vectors.get_vector(token), token in keyed_vectors.key_to_index, True
        except KeyError:
            return None, False, False
    if token in keyed_vectors:
        return keyed_vectors[token], True, True
    return None, False, False


def mean_vector_and_oov_stats(tokens, keyed_vectors, vector_source, weights=None):
    vectors = []
    vector_weights = []
    in_vocab_tokens = []
    subword_tokens = []
    oov_tokens = []
    for token in tokens:
        token_vector, is_explicit_vocab, has_vector = lookup_token_vector(token, keyed_vectors, vector_source)
        if not has_vector:
            oov_tokens.append(token)
            continue
        
        weight = 1.0 if weights is None else weights.get(token, 1.0)
        vectors.append(token_vector * weight)
        vector_weights.append(weight)
        
        if is_explicit_vocab:
            in_vocab_tokens.append(token)
        else:
            subword_tokens.append(token)

    if vectors:
        if weights is not None:
            sentence_vector = (np.sum(vectors, axis=0) / sum(vector_weights)).astype(np.float32)
        else:
            sentence_vector = np.mean(vectors, axis=0).astype(np.float32)
    else:
        sentence_vector = np.zeros(keyed_vectors.vector_size, dtype=np.float32)

    token_count = len(tokens)
    oov_count = len(oov_tokens)
    stats = {
        'token_count': token_count,
        'in_vocab_count': len(in_vocab_tokens),
        'subword_count': len(subword_tokens),
        'oov_count': oov_count,
        'oov_ratio': 0.0 if token_count == 0 else oov_count / token_count,
        'subword_ratio': 0.0 if token_count == 0 else len(subword_tokens) / token_count,
        'in_vocab_tokens': set(in_vocab_tokens),
        'subword_tokens': set(subword_tokens),
        'oov_tokens': set(oov_tokens),
    }
    return sentence_vector, stats


def pair_features_from_vectors(premise, hypothesis, tokenized_frame, aggregation_method, oov_stats=None, oov_feature_mode='none'):
    abs_diff = np.abs(premise - hypothesis)
    product_vec = premise * hypothesis
    
    # Efficient row-wise cosine similarity to avoid MemoryError on large matrices
    norms_p = np.linalg.norm(premise, axis=1, keepdims=True)
    norms_h = np.linalg.norm(hypothesis, axis=1, keepdims=True)
    # Protect against division by zero
    norms_p[norms_p == 0] = 1.0
    norms_h[norms_h == 0] = 1.0
    cosine_vec = np.sum(premise * hypothesis, axis=1, keepdims=True) / (norms_p * norms_h)
    length_features = np.array([
        [len(premise_tokens), len(hypothesis_tokens), len(set(premise_tokens) & set(hypothesis_tokens))]
        for premise_tokens, hypothesis_tokens in zip(
            tokenized_frame['premise_tokens'], tokenized_frame['hypothesis_tokens']
        )
    ], dtype=np.float32)

    if aggregation_method in {'mean_pair', 'tfidf_mean_pair'}:
        blocks = [premise, hypothesis]
    elif aggregation_method == 'diff_product':
        blocks = [abs_diff, product_vec]
    elif aggregation_method == 'cosine_only':
        blocks = [cosine_vec, length_features]
    elif aggregation_method == 'all_features':
        blocks = [premise, hypothesis, abs_diff, product_vec, cosine_vec, length_features]
    else:
        raise ValueError(f'Unknown aggregation method: {aggregation_method}')

    if oov_feature_mode == 'coverage_stats':
        if oov_stats is None:
            raise ValueError('oov_stats are required when oov_feature_mode="coverage_stats".')
        oov_features = np.array([
            [
                premise_stats['oov_count'],
                premise_stats['oov_ratio'],
                hypothesis_stats['oov_count'],
                hypothesis_stats['oov_ratio'],
                abs(premise_stats['oov_ratio'] - hypothesis_stats['oov_ratio']),
                premise_stats['subword_ratio'],
                hypothesis_stats['subword_ratio'],
                len(premise_stats['in_vocab_tokens'] & hypothesis_stats['in_vocab_tokens']),
                len(premise_stats['subword_tokens'] & hypothesis_stats['subword_tokens']),
                len(premise_stats['oov_tokens'] & hypothesis_stats['oov_tokens']),
            ]
            for premise_stats, hypothesis_stats in zip(oov_stats['premise'], oov_stats['hypothesis'])
        ], dtype=np.float32)
        blocks.append(oov_features)
    elif oov_feature_mode != 'none':
        raise ValueError(f'Unknown OOV feature mode: {oov_feature_mode}')

    return np.hstack(blocks)


def build_pretrained_feature_matrices(frames, tokenized_frames, aggregation_method, embedding_mode, weights=None, oov_feature_mode='none'):
    vector_size = 300
    sentence_vectors = {
        split: {
            'premise': np.zeros((len(frame), vector_size), dtype=np.float32),
            'hypothesis': np.zeros((len(frame), vector_size), dtype=np.float32),
        }
        for split, frame in frames.items()
    }
    oov_stats = {
        split: {
            'premise': [None] * len(frame),
            'hypothesis': [None] * len(frame),
        }
        for split, frame in frames.items()
    }

    for lang in PRETRAINED_FASTTEXT_LANGS:
        path = download_fasttext_vectors(lang, embedding_mode) if DOWNLOAD_PRETRAINED_FASTTEXT else fasttext_vector_path(lang, embedding_mode)
        if not path.exists():
            raise FileNotFoundError(
                f'Missing pretrained vectors for {embedding_mode}/{lang}: expected {path}. '
                f'Download from {fasttext_download_url(lang, embedding_mode)} or set DOWNLOAD_PRETRAINED_FASTTEXT = True.'
            )

        print(f'Loading {lang} vectors from {path}')
        keyed_vectors, vector_source = load_fasttext_vectors(path, embedding_mode)

        for split, frame in frames.items():
            mask = frame['lang_abv'].to_numpy() == lang
            if not mask.any():
                continue
            tokenized = tokenized_frames[split]
            premise_rows = tokenized.loc[mask, 'premise_tokens']
            hypothesis_rows = tokenized.loc[mask, 'hypothesis_tokens']
            premise_vectors = []
            premise_stats = []
            for tokens in premise_rows:
                vector, stats = mean_vector_and_oov_stats(tokens, keyed_vectors, vector_source, weights)
                premise_vectors.append(vector)
                premise_stats.append(stats)

            hypothesis_vectors = []
            hypothesis_stats = []
            for tokens in hypothesis_rows:
                vector, stats = mean_vector_and_oov_stats(tokens, keyed_vectors, vector_source, weights)
                hypothesis_vectors.append(vector)
                hypothesis_stats.append(stats)

            sentence_vectors[split]['premise'][mask] = np.vstack(premise_vectors)
            sentence_vectors[split]['hypothesis'][mask] = np.vstack(hypothesis_vectors)

            row_indices = np.flatnonzero(mask)
            for row_index, stats in zip(row_indices, premise_stats):
                oov_stats[split]['premise'][row_index] = stats
            for row_index, stats in zip(row_indices, hypothesis_stats):
                oov_stats[split]['hypothesis'][row_index] = stats

        del keyed_vectors
        gc.collect()

    return {
        split: pair_features_from_vectors(
            sentence_vectors[split]['premise'],
            sentence_vectors[split]['hypothesis'],
            tokenized_frames[split],
            aggregation_method,
            oov_stats=oov_stats[split],
            oov_feature_mode=oov_feature_mode,
        )
        for split in frames
    }

## Model Families

The required five model families are always covered by Logistic Regression, Linear SVM, Random Forest, Decision Tree, and Gaussian Naive Bayes. XGBoost and LightGBM are added automatically when installed.

In [ ]:
def make_model(model_family, model_params):
    if model_family == 'logistic_regression':
        return make_pipeline(
            StandardScaler(),
            LogisticRegression(max_iter=2000, class_weight='balanced', random_state=RANDOM_STATE, **model_params),
        )
    if model_family == 'linear_svm':
        return make_pipeline(
            StandardScaler(),
            LinearSVC(class_weight='balanced', random_state=RANDOM_STATE, dual='auto', **model_params),
        )
    if model_family == 'random_forest':
        return RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=-1, class_weight='balanced', **model_params)
    if model_family == 'decision_tree':
        return DecisionTreeClassifier(random_state=RANDOM_STATE, class_weight='balanced', **model_params)
    if model_family == 'gaussian_nb':
        return make_pipeline(
            StandardScaler(),
            GaussianNB(**model_params),
        )
    if model_family == 'xgboost' and XGBClassifier is not None:
        return XGBClassifier(
            random_state=RANDOM_STATE,
            objective='multi:softprob',
            eval_metric='mlogloss',
            n_jobs=-1,
            **model_params,
        )
    if model_family == 'lightgbm' and LGBMClassifier is not None:
        return LGBMClassifier(random_state=RANDOM_STATE, class_weight='balanced', n_jobs=-1, **model_params)
    raise ValueError(f'Unsupported or unavailable model family: {model_family}')


def make_model_specs(model_param_grids):
    specs = []
    for model_family, param_grid in model_param_grids.items():
        if model_family == 'xgboost' and XGBClassifier is None:
            continue
        if model_family == 'lightgbm' and LGBMClassifier is None:
            continue
        for model_params in ParameterGrid(param_grid):
            specs.append({
                'model_family': model_family,
                'model_params': model_params,
            })
    return specs


BASE_FEATURE_MODEL_SPEC = {
    'model_family': BASE_FEATURE_MODEL_FAMILY,
    'model_params': BASE_FEATURE_MODEL_PARAMS,
}
MODEL_GRID = make_model_specs(MODEL_PARAM_GRIDS)

pd.Series([item['model_family'] for item in MODEL_GRID]).value_counts()

## Build Staged Experiment Grids

Stage 1 compares feature configurations on one fixed baseline model. Stage 2 takes the best Stage 1 feature configuration and searches model families plus model hyperparameters on that feature set.

In [ ]:
def cap_experiments(experiments, max_experiments):
    if max_experiments is None or len(experiments) <= max_experiments:
        return experiments
    stride = max(1, len(experiments) // max_experiments)
    return experiments[::stride][:max_experiments]


FEATURE_GRID = []
for preprocess_params, aggregation_method, oov_feature_mode in product(
    list(ParameterGrid(PREPROCESS_OPTIONS)),
    AGGREGATION_METHODS,
    OOV_FEATURE_MODES,
):
    FEATURE_GRID.append({
        **preprocess_params,
        'embedding_mode': 'vec_lookup',
        'aggregation_method': aggregation_method,
        'oov_feature_mode': oov_feature_mode,
        'uses_tfidf': aggregation_method == 'tfidf_mean_pair',
    })

if ENABLE_BIN_SUBWORD_EXPERIMENTS:
    for aggregation_method in AGGREGATION_METHODS:
        FEATURE_GRID.append({
            'token_strategy': 'raw',
            'stop_strategy': 'keep_all',
            'embedding_mode': 'bin_subword',
            'aggregation_method': aggregation_method,
            'oov_feature_mode': 'none',
            'uses_tfidf': aggregation_method == 'tfidf_mean_pair',
        })

STAGE_1_EXPERIMENTS = cap_experiments([
    {
        **feature_params,
        **BASE_FEATURE_MODEL_SPEC,
        'stage': 'feature_search',
    }
    for feature_params in FEATURE_GRID
], STAGE_1_MAX_FEATURE_EXPERIMENTS)

len(FEATURE_GRID), len(STAGE_1_EXPERIMENTS), len(MODEL_GRID)

## Run Experiments

In [ ]:
def make_run_name(params):
    model_params_text = '_'.join(f'{key}{value}' for key, value in params['model_params'].items())
    return (
        f"{params['stage']}_pretrained_fasttext_{params['embedding_mode']}_"
        f"{params['token_strategy']}_{params['stop_strategy']}_"
        f"{params['aggregation_method']}_oov{params['oov_feature_mode']}_"
        f"{params['model_family']}_{model_params_text}"
    ).replace('.', 'p').replace('None', 'none')[:240]


def log_run_artifacts(run_name, params, metrics, model, submission, tfidf_vectorizer=None):
    metadata = {
        'run_name': run_name,
        'data_source': data_source,
        'serving_type': 'sklearn_pretrained_fasttext_sentence_pair_features',
        'embedding_source': 'fasttext_common_crawl_wikipedia_157_languages',
        'embedding_mode': params['embedding_mode'],
        'label_mapping': {0: 'entailment', 1: 'neutral', 2: 'contradiction'},
        'preprocessing': {
            'tokenizer': 'lowercase_unicode_regex_tokens',
            'token_strategy': params['token_strategy'],
            'stop_strategy': params['stop_strategy'],
            'lemmatizer': 'simplemma',
            'simplemma_supported_dataset_languages': sorted(set(PRETRAINED_FASTTEXT_LANGS) & SIMPLEMMA_SUPPORTED_LANGS),
            'simplemma_unsupported_dataset_languages': SIMPLEMMA_UNSUPPORTED_DATASET_LANGS,
        },
        'stage': params['stage'],
        'feature_order': params['aggregation_method'],
        'oov_feature_mode': params['oov_feature_mode'],
        'pretrained_fasttext': {
            'languages': PRETRAINED_FASTTEXT_LANGS,
            'vector_dir': str(PRETRAINED_FASTTEXT_DIR),
            'vec_urls': PRETRAINED_FASTTEXT_VEC_URLS,
            'bin_urls': PRETRAINED_FASTTEXT_BIN_URLS,
            'embedding_mode': params['embedding_mode'],
            'max_vectors_loaded_per_language': PRETRAINED_FASTTEXT_MAX_VECTORS,
        },
        'experiment_config': EXPERIMENT_CONFIG,
        'params': params,
        'metrics': metrics,
    }

    with TemporaryDirectory() as temp_dir:
        temp_dir = Path(temp_dir)
        model_path = temp_dir / 'model.joblib'
        submission_path = temp_dir / f'{run_name}.csv'
        tfidf_path = temp_dir / 'tfidf.joblib'
        joblib.dump(model, model_path)
        save_submission(submission, submission_path)

        artifacts = {'model.joblib': model_path}
        if tfidf_vectorizer is not None:
            joblib.dump(tfidf_vectorizer, tfidf_path)
            artifacts['tfidf.joblib'] = tfidf_path

        if MLFLOW_LOGGING:
            with start_notebook_run(
                run_name,
                tags={
                    'model_family': params['model_family'],
                    'features': 'pretrained_fasttext_sentence_pair',
                    'stage': params['stage'],
                    'embedding_mode': params['embedding_mode'],
                    'aggregation_method': params['aggregation_method'],
                    'oov_feature_mode': params['oov_feature_mode'],
                },
            ):
                mlflow.log_params({key: value for key, value in params.items() if key != 'model_params'})
                mlflow.log_params({f'model__{key}': value for key, value in params['model_params'].items()})
                mlflow.log_param('data_source', data_source)
                mlflow.log_param('embedding_source', 'fasttext_common_crawl_wikipedia_157_languages')
                mlflow.log_param('embedding_mode', params['embedding_mode'])
                mlflow.log_param('pretrained_languages', ','.join(PRETRAINED_FASTTEXT_LANGS))
                mlflow.log_dict(EXPERIMENT_CONFIG, 'config/experiment_config.json')
                log_metrics(metrics)
                log_common_context('../configs/data_split.yaml', '../data/processed/split_metadata.json')
                mlflow.log_artifact(str(submission_path), artifact_path='submissions')
                log_model_artifacts(artifacts=artifacts, metadata=metadata, artifact_path='model')


required_embedding_modes = sorted({params['embedding_mode'] for params in STAGE_1_EXPERIMENTS})
missing_fasttext_vectors = validate_pretrained_fasttext_files(PRETRAINED_FASTTEXT_LANGS, required_embedding_modes)
if missing_fasttext_vectors and not DOWNLOAD_PRETRAINED_FASTTEXT:
    for embedding_mode, lang, path, url in missing_fasttext_vectors:
        print(f'{embedding_mode}/{lang}: expected {path} or download {url}')
    raise FileNotFoundError(
        'Missing pretrained fastText vectors. Place files under PRETRAINED_FASTTEXT_DIR '
        'or set DOWNLOAD_PRETRAINED_FASTTEXT = True.'
    )

frames = {'train': train_df, 'val': val_df, 'test': test_df}
token_cache = {}
results = []
best_result = None
best_feature_result = None


def feature_key(params):
    return (
        params['token_strategy'],
        params['stop_strategy'],
        params['embedding_mode'],
        params['aggregation_method'],
        params['oov_feature_mode'],
    )


def get_tokenized_frames(params):
    token_key = (params['token_strategy'], params['stop_strategy'])
    if token_key not in token_cache:
        token_cache[token_key] = {
            split: tokenize_frame(frame, params['token_strategy'], params['stop_strategy'])
            for split, frame in frames.items()
        }
    return token_cache[token_key]


def build_features_for_params(params):
    tokenized_frames = get_tokenized_frames(params)

    tfidf_vectorizer = None
    weights = None
    if params['uses_tfidf']:
        tfidf_vectorizer, weights = fit_tfidf_weights(tokenized_frames['train'])

    features = build_pretrained_feature_matrices(
        frames,
        tokenized_frames,
        aggregation_method=params['aggregation_method'],
        embedding_mode=params['embedding_mode'],
        weights=weights,
        oov_feature_mode=params['oov_feature_mode'],
    )
    return features, tfidf_vectorizer


def run_experiment(index, total, params):
    run_name = make_run_name(params)
    print(f'[{index}/{total}] Running {run_name}...')

    features, tfidf_vectorizer = build_features_for_params(params)

    model = make_model(params['model_family'], params['model_params'])
    model.fit(features['train'], train_df['label'].astype(int))
    val_preds = model.predict(features['val'])
    metrics = compute_all_metrics(val_df, val_preds)
    test_preds = model.predict(features['test'])
    submission = build_submission(sample_submission, test_preds)

    row = {
        'run_name': run_name,
        'accuracy': metrics['accuracy'],
        'f1_macro': metrics['f1_macro'],
        'precision_macro': metrics['precision_macro'],
        'recall_macro': metrics['recall_macro'],
        'model_family': params['model_family'],
        'aggregation_method': params['aggregation_method'],
        'token_strategy': params['token_strategy'],
        'stop_strategy': params['stop_strategy'],
        'embedding_source': 'fasttext_common_crawl_wikipedia_157_languages',
        'embedding_mode': params['embedding_mode'],
        'vector_size': 300,
        'stage': params['stage'],
        'oov_feature_mode': params['oov_feature_mode'],
        'model_params': params['model_params'],
    }

    log_run_artifacts(run_name, params, metrics, model, submission, tfidf_vectorizer)

    del model, features, val_preds, test_preds, submission, tfidf_vectorizer
    gc.collect()
    return row


# Run experiments
# Stage-1: consume SKIP_FIRST_N_EXPERIMENTS here first
skip_stage1 = min(SKIP_FIRST_N_EXPERIMENTS, len(STAGE_1_EXPERIMENTS))
for index, params in enumerate(STAGE_1_EXPERIMENTS, start=1):
    if index <= skip_stage1:
        print(f'Skipping Stage-1 experiment [{index}/{len(STAGE_1_EXPERIMENTS)}] due to SKIP_FIRST_N_EXPERIMENTS={SKIP_FIRST_N_EXPERIMENTS}')
        continue
    row = run_experiment(index, len(STAGE_1_EXPERIMENTS), params)
    results.append(row)
    if best_feature_result is None or row['f1_macro'] > best_feature_result['f1_macro']:
        best_feature_result = row
    if best_result is None or row['f1_macro'] > best_result['f1_macro']:
        best_result = row

# compute remaining skip for Stage-2 (if any)
remaining_skip = max(0, SKIP_FIRST_N_EXPERIMENTS - skip_stage1)

if MLFLOW_LOGGING:
    print('Fetching best Stage-1 experiment from MLflow...')
    try:
        from mlflow.entities import RunStatus
        # Query runs from current experiment with stage=feature_search tag
        mlflow.set_tracking_uri("https://dagshub.com/stepan.nazar.23/Contradictory-My-Dear-Watson.mlflow")
        runs_df = mlflow.search_runs(
            experiment_ids=[
                mlflow.get_experiment_by_name(DEFAULT_EXPERIMENT_NAME).experiment_id
            ],
            filter_string="tags.stage = 'feature_search'",
            order_by=["metrics.accuracy DESC"],
            max_results=1,
        )

        if not runs_df.empty:
            best_run = runs_df.iloc[0]

            run_name = best_run.get("tags.mlflow.runName", "unknown")
            accuracy = best_run.get("metrics.accuracy", 0.0)

            print(
                f"Best Stage-1 MLflow run: {run_name} "
                f"(accuracy={accuracy:.4f})"
            )

            # Extract feature params from MLflow run
            aggregation_method = best_run.get("params.aggregation_method")

            best_feature_params = {
                "token_strategy": best_run.get("params.token_strategy"),
                "stop_strategy": best_run.get("params.stop_strategy"),
                "aggregation_method": aggregation_method,
                "embedding_mode": best_run.get("params.embedding_mode"),
                "oov_feature_mode": best_run.get("params.oov_feature_mode"),
                "uses_tfidf": aggregation_method == "tfidf_mean_pair",
            }
        else:
            print('No Stage-1 runs found in MLflow; using in-memory best result.')
            best_feature_params = {
                'token_strategy': best_feature_result['token_strategy'],
                'stop_strategy': best_feature_result['stop_strategy'],
                'aggregation_method': best_feature_result['aggregation_method'],
                'embedding_mode': best_feature_result['embedding_mode'],
                'oov_feature_mode': best_feature_result['oov_feature_mode'],
                'uses_tfidf': best_feature_result['aggregation_method'] == 'tfidf_mean_pair',
            }
    except Exception as e:
        print(f'Warning: failed to fetch best Stage-1 from MLflow: {e}. Using in-memory best result.')
        best_feature_params = {
            'token_strategy': best_feature_result['token_strategy'],
            'stop_strategy': best_feature_result['stop_strategy'],
            'aggregation_method': best_feature_result['aggregation_method'],
            'embedding_mode': best_feature_result['embedding_mode'],
            'oov_feature_mode': best_feature_result['oov_feature_mode'],
            'uses_tfidf': best_feature_result['aggregation_method'] == 'tfidf_mean_pair',
        }
else:
    # MLflow logging disabled; use in-memory best result
    best_feature_params = {
        'token_strategy': best_feature_result['token_strategy'],
        'stop_strategy': best_feature_result['stop_strategy'],
        'aggregation_method': best_feature_result['aggregation_method'],
        'embedding_mode': best_feature_result['embedding_mode'],
        'oov_feature_mode': best_feature_result['oov_feature_mode'],
        'uses_tfidf': best_feature_result['aggregation_method'] == 'tfidf_mean_pair',
    }

print(f'Using best feature configuration: {best_feature_params}')

STAGE_2_EXPERIMENTS = cap_experiments([
    {
        **best_feature_params,
        **model_spec,
        'stage': 'model_search',
    }
    for model_spec in MODEL_GRID
], STAGE_2_MAX_MODEL_EXPERIMENTS)

for index, params in enumerate(STAGE_2_EXPERIMENTS, start=1):
    if index <= remaining_skip:
        print(f'Skipping Stage-2 experiment [{index}/{len(STAGE_2_EXPERIMENTS)}] due to remaining SKIP_FIRST_N_EXPERIMENTS={remaining_skip}')
        continue
    row = run_experiment(index, len(STAGE_2_EXPERIMENTS), params)
    results.append(row)
    if best_result is None or row['f1_macro'] > best_result['f1_macro']:
        best_result = row

results_df = pd.DataFrame(results).sort_values('f1_macro', ascending=False)
results_df.head(20)